# Day 1 -- CSV vs Parquet

Convert `olist_orders_dataset.csv` to Parquet and measure what changes.

The claim to test: columnar storage compresses better, reads faster, and
preserves types. We measure all three rather than taking it on faith.

In [1]:
import time
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV = ROOT / "data" / "olist" / "olist_orders_dataset.csv"
OUT = ROOT / "data" / "parquet"
OUT.mkdir(parents=True, exist_ok=True)

MB = 1024 * 1024


def timeit(fn, n=3):
    """Best of n -- best, not mean, since OS noise only ever adds time."""
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        result = fn()
        times.append(time.perf_counter() - t0)
    return result, min(times)


print(f"source: {CSV.name}  ({CSV.stat().st_size / MB:.1f} MB)")

source: olist_orders_dataset.csv  (16.8 MB)


## 1. Read the CSV

Timestamps are parsed explicitly. A CSV has no type information at all --
every column arrives as text and something has to guess. That guessing is
part of the read cost, and part of why CSV is fragile.

In [2]:
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

df, csv_read_s = timeit(lambda: pd.read_csv(CSV, parse_dates=DATE_COLS))
csv_size = CSV.stat().st_size

print(f"rows        : {len(df):,}")
print(f"columns     : {len(df.columns)}")
print(f"read time   : {csv_read_s:.3f}s")
print(f"file size   : {csv_size / MB:.2f} MB")
df.dtypes

rows        : 99,441
columns     : 8
read time   : 0.414s
file size   : 16.84 MB


order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## 2. Write Parquet at three compression settings

`snappy` is the default and the usual production choice: fast to
decompress, decent ratio. `gzip` trades CPU for size. `none` isolates how
much of the saving is the columnar layout itself rather than the codec.

In [4]:
variants = {}
for codec in ["snappy", "gzip", None]:
    path = OUT / f"orders_{codec or 'uncompressed'}.parquet"
    _, write_s = timeit(lambda p=path, c=codec: df.to_parquet(p, compression=c, index=False), n=1)
    variants[codec or "none"] = {
        "path": path,
        "size": path.stat().st_size,
        "write_s": write_s,
    }
    print(f"{str(codec):12} {path.stat().st_size / MB:6.2f} MB   written in {write_s:.3f}s")

snappy         9.91 MB   written in 0.176s
gzip           7.13 MB   written in 1.341s
None          10.63 MB   written in 0.070s


## 3. Read each back

In [5]:
for name, v in variants.items():
    _, v["read_s"] = timeit(lambda p=v["path"]: pd.read_parquet(p))
    print(f"{name:12} read {v['read_s']:.3f}s")

snappy       read 0.063s
gzip         read 0.076s
none         read 0.046s


## 4. The comparison

In [6]:
rows = [{
    "format": "CSV",
    "size_mb": round(csv_size / MB, 2),
    "vs_csv_size": "1.0x",
    "read_s": round(csv_read_s, 3),
    "vs_csv_read": "1.0x",
}]
for name, v in variants.items():
    rows.append({
        "format": f"Parquet ({name})",
        "size_mb": round(v["size"] / MB, 2),
        "vs_csv_size": f"{csv_size / v['size']:.1f}x smaller",
        "read_s": round(v["read_s"], 3),
        "vs_csv_read": f"{csv_read_s / v['read_s']:.1f}x faster",
    })

comparison = pd.DataFrame(rows)
comparison

,format,size_mb,vs_csv_size,read_s,vs_csv_read
0,CSV,16.84,1.0x,0.414,1.0x
1,Parquet (snappy),9.91,1.7x smaller,0.063,6.6x faster
2,Parquet (gzip),7.13,2.4x smaller,0.076,5.4x faster
3,Parquet (none),10.63,1.6x smaller,0.046,8.9x faster


## 5. Column pruning -- the reason columnar matters

Size and speed above are nice. This is the structural win.

A CSV is row-major: to read one column you must parse every byte of every
row and discard most of it. Parquet stores each column contiguously, so
reading 2 of 8 columns touches only those 2 on disk.

This is why warehouses are columnar. Analytical queries touch few columns
of many rows -- exactly the case row storage handles worst.

In [7]:
# order_status is ~0.1% of the file's bytes. order_id, by contrast, is a
# third of it -- picking that one would hide the effect being measured.
SMALL = ["order_status"]

_, csv_small = timeit(lambda: pd.read_csv(CSV, usecols=SMALL))
_, pq_small = timeit(lambda: pd.read_parquet(variants["snappy"]["path"], columns=SMALL))

print("1 of 8 columns (order_status)")
print(f"  CSV     : {csv_small:.3f}s   ({csv_small / csv_read_s * 100:.0f}% of a full read)")
print(f"  Parquet : {pq_small:.3f}s   ({pq_small / variants['snappy']['read_s'] * 100:.0f}% of a full read)")
print(f"\n  -> {csv_small / pq_small:.1f}x faster")

# For contrast: a column that IS most of the file.
BIG = ["order_id"]
_, csv_big = timeit(lambda: pd.read_csv(CSV, usecols=BIG))
_, pq_big = timeit(lambda: pd.read_parquet(variants["snappy"]["path"], columns=BIG))
print(f"\n1 of 8 columns (order_id -- 32% of the bytes)")
print(f"  CSV     : {csv_big:.3f}s   ({csv_big / csv_read_s * 100:.0f}% of a full read)")
print(f"  Parquet : {pq_big:.3f}s   ({pq_big / variants['snappy']['read_s'] * 100:.0f}% of a full read)")

1 of 8 columns (order_status)
  CSV     : 0.071s   (17% of a full read)
  Parquet : 0.006s   (10% of a full read)

  -> 10.9x faster

1 of 8 columns (order_id -- 32% of the bytes)
  CSV     : 0.094s   (23% of a full read)
  Parquet : 0.025s   (39% of a full read)


Read the percentages, not just the ratios.

CSV costs roughly the same fraction of a full read whichever column you
ask for -- the parser still walks every byte and throws away what you did
not want. Parquet's cost tracks the size of the columns requested: a tiny
column is near-free, a large one is not.

That is the whole argument for columnar storage. It is not that Parquet is
faster in general; it is that Parquet lets you *not read* what you did not
ask for, and analytical queries ask for few columns of many rows.

## 6. Metadata and row groups

Parquet carries a schema and per-row-group statistics (min/max per column).
An engine reading `WHERE order_purchase_timestamp >= '2018-01-01'` can skip
entire row groups whose max is below that -- without decompressing them.
That is predicate pushdown, and CSV cannot do it at any price.

In [ ]:
pf = pq.ParquetFile(variants["snappy"]["path"])
print(f"row groups : {pf.num_row_groups}")
print(f"rows       : {pf.metadata.num_rows:,}")
print(f"created by : {pf.metadata.created_by}")
print("\nper-column, row group 0:")
rg = pf.metadata.row_group(0)
for i in range(rg.num_columns):
    c = rg.column(i)
    name = c.path_in_schema
    ratio = c.total_uncompressed_size / max(c.total_compressed_size, 1)
    print(f"  {name:32} {c.compression:8} {c.total_compressed_size / 1024:8.1f} KB  ({ratio:.1f}x)")

## 7. Types survive the round trip

The quiet advantage. Parquet stores the schema, so a datetime comes back a
datetime. CSV stores text, so every reader re-guesses -- and guesses
differently. This is the same failure mode as `09790` becoming `9790` in
the zip codes: silent, and only noticed much later.

In [ ]:
csv_naive = pd.read_csv(CSV)                                  # no parse_dates
pq_back = pd.read_parquet(variants["snappy"]["path"])

check = pd.DataFrame({
    "original": df.dtypes.astype(str),
    "csv_roundtrip": csv_naive.dtypes.astype(str),
    "parquet_roundtrip": pq_back.dtypes.astype(str),
})
check["csv_ok"] = check.original == check.csv_roundtrip
check["parquet_ok"] = check.original == check.parquet_roundtrip
check

In [ ]:
print(f"columns preserved by CSV     : {check.csv_ok.sum()}/{len(check)}")
print(f"columns preserved by Parquet : {check.parquet_ok.sum()}/{len(check)}")

## Conclusion

| Dimension | Winner | Why it matters |
|---|---|---|
| File size | Parquet | Columnar + per-column encoding; storage and transfer cost |
| Full read | Parquet | No text parsing, no type guessing |
| Partial read | Parquet, decisively | Only requested columns are touched |
| Type safety | Parquet | Schema travels with the data |
| Human-readable | CSV | The one real advantage -- you can open it in an editor |
| Appendable | CSV | Parquet is immutable; you add files, not rows |

Use CSV for interchange with humans and systems you do not control.
Use Parquet for everything inside the pipeline. On Day 5 the Spark marts
are written as Parquet partitioned by month, which is this same argument
applied at the directory level: skip whole files, not just row groups.